In [2]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def analyze_manager_report(file_path):
    # 1. 加载生成的 Markdown 报告
    loader = UnstructuredMarkdownLoader(file_path)
    docs = loader.load()
    report_content = docs[0].page_content

    # 2. 针对新需求优化的 Prompt 模板
    prompt = ChatPromptTemplate.from_messages([
        ("system", """你是一位资深的基金评价专家。请根据提供的报告，完成以下维度的穿透分析，字数严格控制在 250 字以内：

1. 经理画像 (占比20%)：
   请精炼总结学历、专业、履历及证书等个人信息。必须明确指出其身份标签：如“名校背景”、“海归/海外工作经历”。若无相关数据则不提及。单人约 50 字，多人则每人约 25 字。

2. 能力诊断 (占比50%)：
   利用其管理的其他基金近一年周度数据。评估【TM模型-择时排名】与【TM模型-选股排名】，必须引用数据分位点（如“Top 10%”）来支撑观点。分析其业绩增长是源于精准择时，还是稳健的个股挖掘。

3. 策略独特性 (占比30%)：
   结合【SDI (选股驱动力)】指标分析其策略的独特程度。解释其收益与同类的相关性：高 SDI 意味着策略极具独特性/非共识，低相关性则反映了经理在获取 alpha 上的独立性。

评价语需专业且客观，严禁废话，直接给出数据背后的逻辑结论。"""),
        ("user", "以下是待分析的基金经理详细报告数据：\n\n{content}")
    ])

    # 3. 初始化模型
    llm = ChatOpenAI(
            model="deepseek-reasoner",
            openai_api_key="sk-84ec55da76944a109524df873d20d975",
            openai_api_base="https://api.deepseek.com",
        )

    # 4. 构建链
    chain = prompt | llm | StrOutputParser()

    # 5. 执行分析
    analysis_result = chain.invoke({"content": report_content})
    return analysis_result

In [3]:
# 使用示例
result = analyze_manager_report("000005_Analysis_Report.md")
print(result)

**经理画像：**
*   轩璇：13年固收领域经验，工学硕士背景，具备基金从业资格，属金融科班出身。
*   吴翠：14年证券从业经历，曾任信用评级机构副总，硕士研究生，同样为资深金融从业者。

**能力诊断：**
核心业绩源于突出的选股能力。在代表性混合基金中，轩璇（002222）与吴翠（001688）的TM选股排名长期稳居Top 20%甚至Top 10%分位（如近期16.9%、20.5%）。相比之下，择时能力（TMTiming）排名多在50%-80%区间，表现平平甚至偏弱。表明两者均是通过深度的个股挖掘而非精准的市场择时来驱动超额收益。

**策略独特性：**
吴翠在管混合基金（001688）的SDI高达0.79，显示其收益绝大部分由纯粹的选股驱动，策略极具独特性，与市场共识及同类基金的相关性低，获取Alpha的独立性很强。轩璇在混合策略上（SDI 0.50）也展现出一定的选股独特性，但在债券型产品中策略更偏向传统风格暴露。
